# Israeli Music Training Notebook

Trains a NEW diffusion U-Net from scratch on three styles (clean multi-version, no warm-start):
- **version_id = 0** — Slakh-rock (down-weighted)
- **version_id = 1** — Israeli_Artists
- **version_id = 2** — Israeli_Military
- **null token  = 3** (CFG unconditional slot, always equals `n_versions`)

The Slakh-only `slakh_sanity/best_val.pt` checkpoint is preserved as a frozen baseline/reference; it is NOT loaded by this run.

## Prerequisites
1. Slakh tensors + train/val CSVs available on Drive (`version_id=0` data).
2. Israeli_Artists and Israeli_Military tensors uploaded to Drive (run `data_ingest_israeli.ipynb` for each version to verify).
3. Israeli train/val/test CSVs generated by `data_ingest_israeli.ipynb` for both versions.
4. `configs/default.yaml` has `conditioning.n_versions: 3` and `conditioning.null_version_idx: 3` (committed + pushed).

## Cell order
| Cell | Purpose |
|------|---------|
| 1 | Environment setup, Drive mount, path constants |
| 1.5 | Verify data — tensor health check + CSV counts |
| 2 | Build combined manifest (Slakh + Israeli_Artists + Israeli_Military) |
| 3 | (Optional) Overfit-one-batch sanity check |
| 4 | Train — full run |
| 5 | Loss curves (train + val overlaid) |
| 5.5 | Mel progression gallery |
| 6 | Gate check part 1 — val loss |
| 7 | Gate check part 2 — style transfer demo (same piano roll → version 0 vs 1 vs 2) |

---

### §37 cell-header convention

Every code cell in this notebook is preceded by a markdown header that
follows the project standard (ENGINEERING_DECISIONS §37, see
[colab/README.md](README.md)):

> **What this does.** plain-English summary.  
> **Inputs.** files / variables / env read.  
> **Outputs.** files / variables / state written.  
> **Action required.** anything the user must edit, click, approve, or run
> elsewhere. Prefix the heading with **⚠** when the action requires switching
> machines (e.g. F1 backfill needs `basic_pitch_env` locally).  
> **Runtime.** order-of-magnitude (seconds / minutes / hours).

`colab/postprocessing.ipynb` is the reference implementation.


## Cell 1 — Environment setup + lock-file verification

**What this does.** Mounts Drive, clones/pulls the repo, installs the Colab
dependencies (+ `basic-pitch`), builds the per-style `DrivePaths` objects and the
`STYLES` / `STYLE_DATA` registry (3 versions: Slakh + Israeli_Artists +
Israeli_Military), then verifies the reproducibility lock for each ingested
Israeli version — refusing to proceed if a config SHA256 drifted from ingest.

**Inputs.** `GITHUB_TOKEN` Colab secret; `configs/default.yaml`; the
`versions/Israeli_Artists` and `versions/Israeli_Military` lock files + split
CSVs on Drive.

**Outputs.** Combined-run artifacts (checkpoints, logs, combined manifests,
demos) are written under a DEDICATED namespace `Israeli_3style/`
(`checkpoints/Israeli_3style`, `logs/Israeli_3style`,
`versions/Israeli_3style/combined_*.csv`). Kernel globals: `P_ARTISTS`,
`P_MILITARY`, `P_PRIMARY`, `STYLES`, `STYLE_DATA`, `N_VERSIONS`,
`COMBINED_TRAIN_CSV`, `COMBINED_VAL_CSV`, `CKPT_DIR`, `LOG_DIR`, `SLAKH_CKPT`.

**Action required.** Set the `GITHUB_TOKEN` secret. Re-run after every `git push`
so the lock check sees the latest commit.

**Runtime.** ~1–2 min (mostly pip install).

In [ ]:
# ============================================================
# Cell 1 — Environment setup
# ============================================================
import os, sys, subprocess

print('[1/5] Mounting Google Drive ...', flush=True)
from google.colab import drive, userdata
drive.mount('/content/drive')
print('      Drive mounted.', flush=True)

print('[2/5] Reading GITHUB_TOKEN secret ...', flush=True)
_token = userdata.get('GITHUB_TOKEN')
assert _token, 'GITHUB_TOKEN Colab secret missing'
REPO_URL = f'https://{_token}@github.com/galgeva1/MusicProject.git'

REPO_DIR = '/content/MusicProject'
if not os.path.exists(REPO_DIR):
    print('[3/5] Cloning repo ...', flush=True)
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    print('[3/5] Repo exists — resetting + pulling latest ...', flush=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', 'HEAD'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
print('      Repo ready.', flush=True)

%cd {REPO_DIR}
print('[4/5] Installing requirements (pip, quiet) ...', flush=True)
!pip install -q -r requirements_colab.txt
print('      Requirements installed.', flush=True)

sys.path.insert(0, REPO_DIR)

print('[5/5] Resolving centralised paths ...', flush=True)
from paths import DrivePaths, assert_env, verify_lock_file
assert_env(expect='/usr/')        # Colab runtime check

# Build per-style path objects so the training cells can stay symmetric.
# All three versions share ONE SourcePool; each has its own versions/<name>/ tree.
# Slakh paths (slakh_*) are drive_root-relative and version-independent, so we
# read them off whichever path object is convenient.
P_ARTISTS  = DrivePaths.colab(version_name='Israeli_Artists')
P_MILITARY = DrivePaths.colab(version_name='Israeli_Military')

# Primary host for the COMBINED training run: combined manifests, checkpoints and
# logs live under a DEDICATED namespace (clearer than reusing a single style's
# folder, since this run mixes all three styles). This host has no ingest lock of
# its own — data integrity is guarded by the per-version lock checks below.
P_PRIMARY = DrivePaths.colab(version_name='Israeli_3style')
P_PRIMARY.ensure_dirs()

# ---- STYLE REGISTRY — single source of truth --------------------------------
# To add a new style:
#   1. Add an entry here (next version_id = len(STYLES))
#   2. Add its data in STYLE_DATA below
#   3. Bump conditioning.n_versions and null_version_idx in configs/default.yaml
#      to equal N_VERSIONS, then commit + push before re-running Cell 4.
STYLES = {
    0: 'slakh_rock',
    1: 'Israeli_Artists',
    2: 'Israeli_Military',
}
N_VERSIONS = len(STYLES)  # must match conditioning.n_versions in default.yaml

# Per-style data sources for the combined-manifest builder in Cell 2.
# 'processed' is the data_root that the split CSV's segment_path/score_path are
# resolved against (Slakh -> slakh_processed; Israeli versions -> version_dir).
STYLE_DATA = {
    0: dict(train_csv=P_PRIMARY.slakh_train_csv,
            val_csv  =P_PRIMARY.slakh_val_csv,
            processed=P_PRIMARY.slakh_processed),
    1: dict(train_csv=P_ARTISTS.train_csv,
            val_csv  =P_ARTISTS.val_csv,
            processed=P_ARTISTS.version_dir),
    2: dict(train_csv=P_MILITARY.train_csv,
            val_csv  =P_MILITARY.val_csv,
            processed=P_MILITARY.version_dir),
}

# Combined manifests (multi-style training input) — hosted under the primary.
COMBINED_TRAIN_CSV = P_PRIMARY.combined_train_csv
COMBINED_VAL_CSV   = P_PRIMARY.combined_val_csv

# Training outputs
CKPT_DIR  = P_PRIMARY.checkpoints_dir
LOG_DIR   = P_PRIMARY.logs_dir
SLAKH_CKPT = P_PRIMARY.slakh_ckpt  # optional warm-start (may not exist)

# ---- Lock-file verification (refuses to train if inputs drifted) ------------
# Verify EACH ingested Israeli version against its own lock. Slakh (version_id 0)
# is fixed and has no per-version lock; the combined host (P_PRIMARY) is an output
# namespace with no lock of its own. A git_commit diff is a soft warning; only a
# version_spec / default_config hash drift is fatal under strict=True.
for P in (P_ARTISTS, P_MILITARY):
    print('\nVerifying reproducibility lock for', P.version_name)
    verify_lock_file(P, strict=True)
    print('  ✓ git commit + config SHA256 match the ingest lock')

print()
print(P_PRIMARY.summary())
print(f'N_VERSIONS   : {N_VERSIONS}')
print(f'Styles       : {STYLES}')

## Cell 1.5 — Verify data: tensor health + CSV counts

**What this does.** Samples mel `.pt` tensors for each style and checks
shape / contiguity / storage bloat, then prints row counts for every style's
`train` and `val` split CSV.

**Inputs.** `STYLES`, `STYLE_DATA` (processed dirs + split CSVs) from Cell 1.

**Outputs.** Console only.

**Action required.** None — confirm tensors are OK (bloat < 2x, contiguous)
and that every split CSV has a non-zero row count.

**Runtime.** seconds.


In [ ]:
# ============================================================
# Cell 1.5 — Verify data: tensor health check + CSV counts
# ============================================================
import pathlib, random, torch, csv

def check_tensors(processed_dir, label, n_sample=5):
    """Sample mel .pt files and check contiguity + shape."""
    processed_dir = pathlib.Path(processed_dir)
    pt_files = list(processed_dir.rglob('**/mels/segment_*.pt'))
    if not pt_files:
        # legacy flat naming
        pt_files = list(processed_dir.rglob('*_mel.pt'))
    if not pt_files:
        print(f'  [{label}] WARNING: no mel .pt files found in {processed_dir}')
        return
    print(f'  [{label}] {len(pt_files)} mel .pt files found')
    for p in random.sample(pt_files, min(n_sample, len(pt_files))):
        t = torch.load(p, weights_only=True)
        storage_size = t.storage().nbytes()
        contiguous_size = t.numel() * t.element_size()
        bloat = storage_size / max(contiguous_size, 1)
        flag = '  OK' if bloat < 2.0 else '  ⚠ BLOATED'
        print(f'    {p.name}  shape={tuple(t.shape)}  contiguous={t.is_contiguous()}  '
              f'bloat={bloat:.1f}x{flag}')

print('=== Tensor health check ===')
for vid, name in STYLES.items():
    check_tensors(STYLE_DATA[vid]['processed'], name)

print()
print('=== Split CSV counts ===')
for vid, name in STYLES.items():
    for split, csv_path in [('train', STYLE_DATA[vid]['train_csv']),
                            ('val',   STYLE_DATA[vid]['val_csv'])]:
        p = pathlib.Path(csv_path)
        if p.exists():
            with open(p, newline='') as f:
                rows = sum(1 for _ in csv.DictReader(f))
            print(f'  {name:15s} {split:5s}: {rows:>5} rows  ({p})')
        else:
            print(f'  {name:15s} {split:5s}: MISSING  ({p})')


## Cell 2 — Build combined train/val manifest

**What this does.** Merges every style's `train` / `val` CSV into
`combined_train.csv` + `combined_val.csv`, stamping each row with its
`version_id` and `data_root` so the training loop can resolve tensors per style.

**Inputs.** `STYLE_DATA` train/val CSVs (Slakh `version_id=0`, Israeli
`version_id=1`).

**Outputs.** `COMBINED_TRAIN_CSV`, `COMBINED_VAL_CSV` files; `train_rows` /
`val_rows`; per-version row counts printed.

**Action required.** None.

**Runtime.** seconds.


In [ ]:
# ============================================================
# Cell 2 — Build combined train/val manifest
# Merges all styles defined in STYLES (Cell 1) into
# combined_train.csv + combined_val.csv.
# Adding a new style = add it to STYLES + STYLE_DATA in Cell 1,
# then re-run this cell.
#
# The two styles live under DIFFERENT roots (Slakh under slakh_processed,
# Israeli under version_dir), but MelPianoRollDataset + the data-quality gate
# resolve every row against a SINGLE manifest_root. To keep the combined
# manifest self-contained, we rewrite segment_path/score_path to ABSOLUTE paths
# here (joined against each style's own root), so resolution is root-independent.
#
# The per-style CSVs may have DIFFERENT columns (e.g. Israeli has 'aug_tag' from
# augmentation, Slakh does not). We take the UNION of all columns and fill any
# missing values with '' so DictWriter accepts every row.
# ============================================================
import csv
from collections import Counter
from pathlib import Path

def merge_manifests(inputs, output_path):
    """Merge CSVs; each input is (csv_path, version_id, data_root).

    segment_path/score_path are rewritten to absolute paths against data_root.
    Output columns = union of all input columns + version_id + data_root.
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # First pass: read every row and collect the union of column names.
    rows_out = []
    seen_cols = []  # preserve first-seen order for a stable header
    for csv_path, version_id, data_root in inputs:
        data_root = Path(data_root)
        with open(csv_path, newline='') as f:
            reader = csv.DictReader(f)
            for col in reader.fieldnames:
                if col not in seen_cols:
                    seen_cols.append(col)
            for row in reader:
                # Path(root) / abs == abs, so this is idempotent on re-runs.
                row['segment_path'] = str(data_root / row['segment_path'])
                row['score_path']   = str(data_root / row['score_path'])
                row['version_id'] = str(version_id)
                row['data_root']  = str(data_root)
                rows_out.append(row)

    fieldnames = seen_cols + [c for c in ('version_id', 'data_root')
                              if c not in seen_cols]

    with open(output_path, 'w', newline='') as f:
        # restval='' fills columns a given row doesn't have (e.g. Slakh rows
        # have no 'aug_tag'); extrasaction='ignore' guards against stragglers.
        writer = csv.DictWriter(f, fieldnames=fieldnames,
                                restval='', extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows_out)
    print(f'Written {len(rows_out)} rows → {output_path}')
    return rows_out

train_inputs = [(STYLE_DATA[v]['train_csv'], v, STYLE_DATA[v]['processed'])
                for v in sorted(STYLES)]
val_inputs   = [(STYLE_DATA[v]['val_csv'],   v, STYLE_DATA[v]['processed'])
                for v in sorted(STYLES)]

train_rows = merge_manifests(train_inputs, COMBINED_TRAIN_CSV)
val_rows   = merge_manifests(val_inputs,   COMBINED_VAL_CSV)

for split_name, rows in [('train', train_rows), ('val', val_rows)]:
    counts = Counter(r['version_id'] for r in rows)
    summary = '  '.join(f'version_id {v} ({STYLES[int(v)]})={counts[str(v)]}'
                        for v in sorted(STYLES))
    print(f'  {split_name}: {summary}  total={len(rows)}')


# Section 2 — Pre-training data-quality gate

This section is **blocking**: training will not start unless every required
check passes. WARN-level results (e.g. RMS distribution, MFCC consistency)
are surfaced but do not block.

Artifacts produced:
- `gate_report.html` — overall status table
- `_assets/gate_report.png` — slide-friendly status table
- `_assets/dataset_stats.png`, `mel_grid.png`, `piano_roll_grid.png`,
  `segment_length_hist.png`, `mfcc_similarity.png`

These are auto-bundled into `deliverables/01_data_quality/` by
`build_deliverables.py`.


## Section 2.1 — Load combined manifest into a DataFrame

**What this does.** Loads the combined train + val CSVs into a single `gate_df`
(tagged with `__split`) and derives `gate_manifest_root` for path resolution, so
the gate sees the real union of the training set.

**Inputs.** `COMBINED_TRAIN_CSV`, `COMBINED_VAL_CSV`.

**Outputs.** `gate_df`, `gate_manifest_root`; row total + per-version counts
printed.

**Action required.** None.

**Runtime.** seconds.


In [ ]:
## Section 2.1 — Load combined train+val into a single DataFrame for the gate
# The gate checks the union of splits so duration/version counts reflect the
# real training set, not just one split.
import pandas as pd

_gate_parts = []
for _name, _path in [('train', COMBINED_TRAIN_CSV), ('val', COMBINED_VAL_CSV)]:
    _df = pd.read_csv(_path)
    _df['__split'] = _name
    _gate_parts.append(_df)
gate_df = pd.concat(_gate_parts, ignore_index=True)

# The gate resolves relative paths against this root. Per-row 'data_root'
# is the absolute root used in Cell 2's merge_manifests().
import pathlib as _p
gate_manifest_root = _p.Path(gate_df.iloc[0]['data_root'])
print(f'Combined manifest: {len(gate_df)} rows  (root: {gate_manifest_root})')
print(gate_df['version_id'].value_counts().to_dict())


## Section 2.2 — Run the data-quality gate (per Israeli style)

**What this does.** Runs the shared full data-quality gate once per NEW Israeli
style (`version_id 1` = Israeli_Artists, `version_id 2` = Israeli_Military) —
required columns, expected `version_id`, minimum hours, RMS / MFCC checks — and
writes HTML + PNG + JSON reports for each style.

**Inputs.** `gate_df`, `gate_manifest_root`, `GATE_VERSIONS={1,2}`,
`MIN_HOURS=3.0`.

**Outputs.** `gate_reports` (dict keyed by version_id) + `gate_report` (the
primary style, for §2.6); per-style `gate_report_<style>.html` / `.png` / `.json`
plus the canonical `gate_report.html` / `.png` / `.json` (primary style, for
build_deliverables.py) under `CKPT_DIR/data_quality/`.

**Action required.** None — review the printed summary per style (FAILs block in §2.6).

**Runtime.** ~2–5 min (gate runs twice).

In [ ]:
## Section 2.2 — Run the data-quality gate (per Israeli style)
# Same gate every style version uses. We run it once per NEW Israeli style
# (version_id 1 = Israeli_Artists, version_id 2 = Israeli_Military) so BOTH
# ingested styles are validated before training. Output per style: a GateReport
# (printed below, also saved as HTML + PNG table + JSON for deliverables).
from pathlib import Path
from preprocessing.data_quality import run_full_gate

# Styles validated by the gate. Slakh (version_id 0) is the frozen baseline whose
# manifest intentionally lists each segment twice — that is not a defect of the
# Israeli data, so it is NOT gated here.
GATE_VERSIONS = {1: 'Israeli_Artists', 2: 'Israeli_Military'}
PRIMARY_GATE_VID = 1  # canonical gate_report.* picked up by build_deliverables.py
MIN_HOURS = 3.0

# Output dir for gate artifacts — also picked up by build_deliverables.py later.
GATE_DIR = Path(CKPT_DIR) / 'data_quality'
GATE_ASSETS = GATE_DIR / '_assets'
GATE_ASSETS.mkdir(parents=True, exist_ok=True)

# Run the gate for each style and collect the reports keyed by version_id.
gate_reports = {}
for _vid, _vname in GATE_VERSIONS.items():
    # The gate is a per-style READINESS check (expected_version_id + min_hours),
    # so it runs on that style's rows only. gate_df stays intact for the
    # combined-set visualization panels in Sections 2.3-2.6.
    _df_style = gate_df[gate_df['version_id'].astype(int) == _vid].reset_index(drop=True)
    print(f'\n=== Gate: {_vname} (version_id={_vid}) ===')
    print(f'Gate input: {len(_df_style)} rows for version_id={_vid} '
          f'(filtered from {len(gate_df)} combined rows)')

    _rep = run_full_gate(
        _df_style, gate_manifest_root,
        expected_version_id=_vid,
        min_hours=MIN_HOURS,
    )
    _rep.print_summary()

    # Per-style artifacts (suffixed so the two runs don't overwrite each other).
    _title = f'{_vname} — Data Quality Gate'
    _rep.write_html(GATE_DIR / f'gate_report_{_vname}.html', title=_title)
    _rep.write_png_table(GATE_ASSETS / f'gate_report_{_vname}.png', title=_title)
    _rep.write_json(GATE_ASSETS / f'gate_report_{_vname}.json')
    print(f'Saved: {GATE_DIR / f"gate_report_{_vname}.html"}')

    gate_reports[_vid] = _rep

# Canonical report (primary style) for build_deliverables.py, which expects the
# un-suffixed gate_report.* filenames. `gate_report` also stays defined for the
# BLOCKING decision in Section 2.6.
gate_report = gate_reports[PRIMARY_GATE_VID]
_primary_name = GATE_VERSIONS[PRIMARY_GATE_VID]
_primary_title = f'{_primary_name} — Data Quality Gate'
gate_report.write_html(GATE_DIR / 'gate_report.html', title=_primary_title)
gate_report.write_png_table(GATE_ASSETS / 'gate_report.png', title=_primary_title)
gate_report.write_json(GATE_ASSETS / 'gate_report.json')
print(f'\nCanonical (deliverables) report: {GATE_DIR / "gate_report.html"} '
      f'({_primary_name})')

## Section 2.3 — Dataset stats panel

**What this does.** Renders the dataset stats panel (segments, hours, versions,
splits) for the combined set.

**Inputs.** `gate_df`.

**Outputs.** `dataset_stats.png` under `CKPT_DIR/data_quality/_assets/`.

**Action required.** None.

**Runtime.** seconds.


In [ ]:
## Section 2.3 — Dataset stats panel (segments, hours, versions, splits)
from preprocessing.dataset_visualizations import plot_dataset_stats_panel
plot_dataset_stats_panel(gate_df, save_path=str(GATE_ASSETS / 'dataset_stats.png'))
print(f'Saved: {GATE_ASSETS / "dataset_stats.png"}')


## Section 2.4 — Sample mel + piano-roll grids

**What this does.** Plots an 8-sample grid of mel spectrograms and piano rolls
for visual inspection of the model inputs/targets.

**Inputs.** `gate_df`, `gate_manifest_root`.

**Outputs.** `mel_grid.png`, `piano_roll_grid.png` under
`CKPT_DIR/data_quality/_assets/`.

**Action required.** None.

**Runtime.** seconds.


In [ ]:
## Section 2.4 — Sample mel + piano-roll grids
from preprocessing.dataset_visualizations import plot_mel_grid, plot_piano_roll_grid

plot_mel_grid(gate_df, gate_manifest_root, n=8,
              save_path=str(GATE_ASSETS / 'mel_grid.png'))
plot_piano_roll_grid(gate_df, gate_manifest_root, n=8,
                     save_path=str(GATE_ASSETS / 'piano_roll_grid.png'))
print('Saved: mel_grid.png, piano_roll_grid.png')


## Section 2.5 — Segment-length histogram + MFCC similarity heatmap

**What this does.** Segment-length histogram (expect a single spike at 430
frames) plus a per-song MFCC cosine-similarity heatmap as a coarse stylistic
coherence / duplicate check (dark off-diagonal cells = outlier songs).

**Inputs.** `gate_df`, `gate_manifest_root`.

**Outputs.** `segment_length_hist.png`, `mfcc_similarity.png` under
`CKPT_DIR/data_quality/_assets/`.

**Action required.** None.

**Runtime.** ~1 min.


In [ ]:
## Section 2.5 — Segment-length histogram + per-song MFCC similarity heatmap
# Histogram should be a single spike at 430. The heatmap is a coarse stylistic
# coherence check: dark off-diagonal cells = outlier songs.
from preprocessing.dataset_visualizations import (
    plot_segment_length_histogram,
    plot_mfcc_similarity_heatmap,
)

plot_segment_length_histogram(gate_df, gate_manifest_root,
                              save_path=str(GATE_ASSETS / 'segment_length_hist.png'))
plot_mfcc_similarity_heatmap(gate_df, gate_manifest_root,
                             save_path=str(GATE_ASSETS / 'mfcc_similarity.png'))
print('Saved: segment_length_hist.png, mfcc_similarity.png')


## Section 2.6 — Gate decision (BLOCKING)

**What this does.** Asserts the gate passed for EVERY validated Israeli style
(`gate_reports` from §2.2) — raises if any check returned FAIL.
WARN-level results are allowed. This is the hard stop before training.

**Inputs.** `gate_reports`.

**Outputs.** Console; raises `AssertionError` on failure.

**Action required.** Must pass to proceed. Only comment this cell out if you
explicitly accept the risk of training on a flagged set.

**Runtime.** seconds.

In [ ]:
## Section 2.6 — Gate decision (BLOCKING)
# Raises if any check returned FAIL. WARNs are allowed (see reports above).
# Asserts EVERY validated Israeli style (see gate_reports from Section 2.2) so a
# failure in either Israeli_Artists or Israeli_Military blocks training.
# Comment this cell out only if you accept the risk of training on a flagged set.
for _vid, _rep in gate_reports.items():
    _rep.assert_pass()
    print(f'  version_id={_vid} ({GATE_VERSIONS[_vid]}): gate PASSED')
print('Data-quality gate PASSED for all styles — safe to start training.')

## Cell 3 — (Optional) Overfit-one-batch sanity check

**What this does.** Smoke test — overfits a single batch for 5000 steps (LR
pinned to peak) to confirm the `n_versions=3` model wires up and has the capacity
to drive loss → ~0 before committing to the full run. Pass/fail uses a 50-step
trailing average (per-step loss is noisy because the diffusion timestep is
sampled at random each step). Target: loss < 0.05.

**Inputs.** `configs/default.yaml`, `COMBINED_TRAIN_CSV`, `COMBINED_VAL_CSV`;
writes to `CKPT_DIR/overfit_test`.

**Outputs.** Throwaway overfit checkpoint dir; final trailing-avg loss printed.

**Action required.** None — this is the go/no-go check. A steady downward loss
curve already confirms the model is wired correctly; the < 0.05 target just
confirms it fully memorised the batch. If the loss plateaus high (e.g. stuck
near ~0.8) or goes NaN, stop and debug config/data before the full run.

**Runtime.** ~5–10 min on GPU.

In [ ]:
# ============================================================
# Cell 3 — (Optional) Overfit-one-batch sanity check
# Quick test that the n_versions=3 model trains without errors.
# Overfits ONE batch with LR pinned to peak; budget set via
# --max_steps. CFG dropout + random-t make single-batch
# memorisation noisy, so the pass/fail uses a trailing avg and
# needs a few thousand steps to comfortably cross < 0.05.
# Output is streamed live (python -u + line-buffered Popen).
# ============================================================

import subprocess, sys

def stream_subprocess(cmd):
    """Run cmd, streaming the child's stdout into the notebook live.
    Uses python -u (unbuffered) + line-by-line Popen so Colab shows
    per-step training output instead of swallowing it."""
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    return proc.returncode

cmd = [
    sys.executable, '-u', 'train.py',
    '--config', 'configs/default.yaml',
    '--ckpt_dir', str(CKPT_DIR) + '/overfit_test',
    '--train_manifest', str(COMBINED_TRAIN_CSV),
    '--val_manifest', str(COMBINED_VAL_CSV),
    '--overfit-one-batch',
    '--max_steps', '5000',
]
# NOTE: configs/default.yaml must have conditioning.n_versions=3 before running this
print('Running overfit test — should converge to loss < 0.05 ...\n', flush=True)
rc = stream_subprocess(cmd)
print(f'\nExit code: {rc}')
if rc != 0:
    print('ERROR: overfit test failed. Check config and data paths.')

## Cell 3.5 — Stage dataset to local SSD (fixes slow Drive reads)

**Why.** Google Drive's FUSE mount has ~100 ms latency *per file* (see
`ENGINEERING_DECISIONS.md` §9.4). Training reads a fresh batch every step, so
reading `.pt` segments straight from Drive starves the GPU (step 0 prints, then
long stalls). The fix is the same one used for Slakh: **copy the segments once
to Colab's local NVMe SSD (~0.1 ms/file) and train from there.**

**What this does.**
- Reads `COMBINED_TRAIN_CSV` + `COMBINED_VAL_CSV`.
- Copies only the *referenced* `.pt` files from each Drive `data_root` to
  `/content/data_local/...` in a single streaming `tar` pass (much faster than
  per-file copy over FUSE).
- Rewrites the manifests so every path points at the local mirror, writes them
  to `/content`, and repoints `COMBINED_TRAIN_CSV` / `COMBINED_VAL_CSV`.

**Inputs.** `COMBINED_TRAIN_CSV`, `COMBINED_VAL_CSV`, and the Drive `data_root`
folders those manifests reference.

**Outputs.** A local mirror of the referenced `.pt` files under
`/content/data_local/...`, rewritten manifests in `/content`, a `.staged_ok`
marker per root, and `COMBINED_TRAIN_CSV` / `COMBINED_VAL_CSV` repointed at the
local copies.

**Idempotent / resume-safe.** Local disk is wiped on disconnect. Just re-run
Cell 1 then this cell before Cell 4 — already-staged roots are detected via a
`.staged_ok` marker and skipped.

**Action required.** Run this **before** Cell 4. Checkpoints still go to Drive
(`CKPT_DIR`), so resume is unaffected.

**Runtime.** a few minutes (one-time copy), then training runs at full speed.

In [ ]:
# ============================================================
# Cell 3.5 - Stage dataset to local SSD (Drive FUSE -> /content NVMe)
# Strategy: tar each data_root from Drive, cache on Drive for reuse,
# then copy+extract to local SSD so training does big sequential reads
# instead of ~46k slow per-file FUSE opens.
#
# Large roots (e.g. a 13 GB Israeli military version dir) are built in
# many small per-album CHUNKS, each with a stall timeout + retries and
# cached individually on Drive. A FUSE stall on one album no longer
# hangs the whole 13 GB build, and a re-run RESUMES from chunks already
# published instead of restarting from zero.
# Idempotent: .staged_ok marker per root; safe to re-run.
# ============================================================

import csv, os, shutil, subprocess, time
from pathlib import Path
from collections import defaultdict

LOCAL_BASE = Path('/content/data_local')
LOCAL_BASE.mkdir(parents=True, exist_ok=True)

_src_train = Path(COMBINED_TRAIN_CSV)
_src_val   = Path(COMBINED_VAL_CSV)

def _read_rows(csv_path):
    with open(csv_path, newline='') as f:
        r = csv.DictReader(f)
        return r.fieldnames, list(r)

tr_fields, tr_rows = _read_rows(_src_train)
va_fields, va_rows = _read_rows(_src_val)
all_rows = tr_rows + va_rows

# --- Map each Drive data_root -> a local mirror dir ------------------------
roots = sorted({row['data_root'] for row in all_rows})
root_to_local = {}
for i, root in enumerate(roots):
    root_to_local[root] = LOCAL_BASE / f'root{i}'
    print(f'  data_root[{i}]: {root}')
    print(f'            -> {root_to_local[root]}')

# Cache dir for reusable tars (on Drive, persists across sessions).
drive_common = Path(os.path.commonpath([str(r) for r in roots]))
DRIVE_TAR_DIR = drive_common / '_stage_tars'
DRIVE_TAR_DIR.mkdir(parents=True, exist_ok=True)
print(f'  tar cache : {DRIVE_TAR_DIR}')

def _run(cmd, **kw):
    """Run cmd, capture text output."""
    return subprocess.run(cmd, capture_output=True, text=True, **kw)

def _chunk_relpaths(src_root, max_depth=3):
    """Split src_root into reasonably-sized chunks for resilient taring.

    Descends directories up to max_depth so a 13 GB tree becomes many small
    per-album chunks instead of one monolithic archive. Each chunk is tarred,
    timed-out, retried and cached independently. Loose files at any level
    become their own chunks. Returns a stable sorted list of paths relative
    to src_root."""
    src_root = Path(src_root)
    chunks = []
    def _rec(d, depth):
        entries = sorted(d.iterdir())
        subdirs = [e for e in entries if e.is_dir()]
        files   = [e for e in entries if e.is_file()]
        if not subdirs or depth >= max_depth:
            chunks.append(d.relative_to(src_root))
            return
        for f in files:
            chunks.append(f.relative_to(src_root))
        for sd in subdirs:
            _rec(sd, depth + 1)
    _rec(src_root, 0)
    return chunks

def _tar_chunk(src_root, relpath, out_tar, timeout=1800, retries=4):
    """Tar one chunk (file or subdir) from Drive -> out_tar.

    A frozen Drive-FUSE read is killed after `timeout`s and retried instead of
    hanging forever (the failure mode that stalled the monolithic build)."""
    for attempt in range(1, retries + 1):
        t0 = time.time()
        try:
            cp = subprocess.run(
                ['tar', '-C', str(src_root), '--ignore-failed-read',
                 '-cf', str(out_tar), str(relpath)],
                capture_output=True, text=True, timeout=timeout)
        except subprocess.TimeoutExpired:
            print(f'      [stall] {relpath} timed out after {timeout}s '
                  f'(attempt {attempt}/{retries}) - retrying')
            Path(out_tar).unlink(missing_ok=True)
            continue
        # rc 0 = ok; rc 1 = warnings only (e.g. 'file changed as we read it'
        # on Drive FUSE) - archive is still valid, accept it. rc>=2 = fatal.
        if cp.returncode <= 1:
            if cp.returncode == 1 and cp.stderr.strip():
                print(f'      [info] tar warning for {relpath}: '
                      f'{cp.stderr.strip().splitlines()[-1]}')
            return time.time() - t0
        print(f'      [warn] tar rc={cp.returncode} for {relpath} '
              f'(attempt {attempt}/{retries})')
        if cp.stderr.strip():
            print('      ', cp.stderr.strip().splitlines()[-1])
        Path(out_tar).unlink(missing_ok=True)
    raise RuntimeError(f'tar chunk failed after {retries} attempts: {relpath}')

def _extract_tar(local_tar, local_root):
    local_root.mkdir(parents=True, exist_ok=True)
    cp = _run(['tar', '-C', str(local_root), '-xf', str(local_tar)])
    if cp.returncode != 0:
        print(cp.stderr[-3000:])
        raise RuntimeError(f'tar extract failed for {local_tar}')

def _stage_root(i, src_root, local_root):
    marker = local_root / '.staged_ok'
    if marker.exists():
        print(f'  [skip] {local_root} already staged')
        return
    local_root.mkdir(parents=True, exist_ok=True)

    # Back-compat fast path: a single monolithic root{i}.tar already published
    # on Drive (roots staged before chunking) -> copy + extract as before.
    legacy_tar = DRIVE_TAR_DIR / f'root{i}.tar'
    if legacy_tar.exists() and legacy_tar.stat().st_size > 1_000_000:
        local_tar = Path(f'/content/_root{i}.tar')
        t0 = time.time()
        print(f'  [cache] copying {legacy_tar} -> {local_tar}')
        shutil.copy(str(legacy_tar), str(local_tar))
        print(f'          copied {local_tar.stat().st_size/1e9:.2f} GB '
              f'in {time.time()-t0:.0f}s')
        _extract_tar(local_tar, local_root)
        local_tar.unlink(missing_ok=True)
        marker.write_text('ok')
        return

    # Chunked path: build/cache per-album tars. A stall only affects one chunk
    # (timed out + retried); a re-run resumes from chunks already on Drive.
    drive_parts = DRIVE_TAR_DIR / f'root{i}_parts'
    drive_parts.mkdir(parents=True, exist_ok=True)
    local_parts = Path(f'/content/_root{i}_parts')
    local_parts.mkdir(parents=True, exist_ok=True)

    chunks = _chunk_relpaths(src_root)
    print(f'  [build] {len(chunks)} chunk(s) for {src_root}')
    t_root = time.time()
    for k, rel in enumerate(chunks):
        part_name = f'part_{k:04d}.tar'
        drive_part = drive_parts / part_name
        local_part = local_parts / part_name
        if drive_part.exists() and drive_part.stat().st_size > 0:
            # Already built on a previous run -> reuse (copy only if needed).
            if not (local_part.exists()
                    and local_part.stat().st_size == drive_part.stat().st_size):
                shutil.copy(str(drive_part), str(local_part))
            _extract_tar(local_part, local_root)
            print(f'    [{k+1}/{len(chunks)}] {rel}  (cached)')
            continue
        dt = _tar_chunk(src_root, rel, local_part)
        _extract_tar(local_part, local_root)
        shutil.copy(str(local_part), str(drive_part))   # publish for next time
        print(f'    [{k+1}/{len(chunks)}] {rel}  '
              f'{local_part.stat().st_size/1e9:.2f} GB in {dt:.0f}s')
    print(f'          staged {len(chunks)} chunk(s) in {time.time()-t_root:.0f}s')
    marker.write_text('ok')

for i, root in enumerate(roots):
    _stage_root(i, root, root_to_local[root])

# --- Rewrite manifests to local paths --------------------------------------
def _rewrite(rows, fields, out_path):
    out_path = Path(out_path)
    for row in rows:
        root = row['data_root']
        local_root = root_to_local[root]
        for col in ('segment_path', 'score_path'):
            rel = Path(row[col]).relative_to(root)
            row[col] = str(local_root / rel)
        row['data_root'] = str(local_root)
    with open(out_path, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fields, restval='',
                           extrasaction='ignore')
        w.writeheader()
        w.writerows(rows)
    return out_path

LOCAL_TRAIN_CSV = _rewrite(tr_rows, tr_fields, '/content/combined_train_local.csv')
LOCAL_VAL_CSV   = _rewrite(va_rows, va_fields, '/content/combined_val_local.csv')

# --- Verify referenced files resolve locally -------------------------------
import random
missing = []
for row in random.sample(all_rows, min(50, len(all_rows))):
    for col in ('segment_path', 'score_path'):
        if not Path(row[col]).exists():
            missing.append(row[col])
if missing:
    print(f'\n  WARNING: {len(missing)} sampled file(s) missing locally, e.g.:')
    for m in missing[:5]:
        print('   ', m)
    raise AssertionError('Some staged files are missing - check tar build above.')
print('  Verified: sampled local files all present.')

# --- Repoint the variables Cell 4 reads ------------------------------------
COMBINED_TRAIN_CSV = LOCAL_TRAIN_CSV
COMBINED_VAL_CSV   = LOCAL_VAL_CSV
print(f'\nStaged. COMBINED_TRAIN_CSV -> {COMBINED_TRAIN_CSV}')
print(f'        COMBINED_VAL_CSV   -> {COMBINED_VAL_CSV}')
print('Now run Cell 4 (it will read from local SSD).')

## Cell 4 — Training (full run)

**What this does.** Trains a NEW diffusion U-Net from scratch on the combined
Slakh + Israeli manifest (no warm-start). Checkpoints, TensorBoard logs, periodic
validation samples and mel snapshots are written throughout.

**Inputs.** `configs/default.yaml` (`total_steps=250000`,
`conditioning.n_versions=3`), `COMBINED_TRAIN_CSV`, `COMBINED_VAL_CSV`.

**Outputs.** `best_val.pt` + step checkpoints in `CKPT_DIR`; event logs in
`LOG_DIR`; sample mels + snapshots.

**Action required.** ⚠ Ensure `conditioning.n_versions=3` and
`null_version_idx=3` are committed + pushed and Cell 1 re-run first. Resumable —
re-running picks up from the latest checkpoint.

**Runtime.** hours (~17h on an A100 for 250k steps).

In [ ]:
# ============================================================
# Cell 4 - Training (clean Israeli v1 multi-version run)
# Trains a NEW model from scratch on the combined Slakh+Israeli
# manifest. No warm-start from the Slakh-only checkpoint - the
# Slakh `best_val.pt` is preserved as a frozen baseline/reference.
#
# AUTO-RESUME: this cell always passes `--resume_from auto`. On the
# first run no checkpoint exists yet, so training starts from scratch.
# If Colab disconnects, just re-run Cell 1 then this cell again: it
# finds the latest step_*.pt in CKPT_DIR and restores model + EMA +
# optimizer + scheduler (so the LR schedule continues correctly) +
# RNG state, then keeps going. Nothing to edit by hand.
#
# BEFORE RUNNING:
#   1. configs/default.yaml has conditioning.n_versions=3 and
#      conditioning.null_version_idx=3 (committed + pushed).
#   2. Re-run Cell 1 (git pull) to pick up the updated config.
#
# A future warm-start / embedding-expansion path is intentionally
# NOT used here; see ENGINEERING_DECISIONS.md / REMAINING_WORKPLAN.md
# for the rationale.
# ============================================================

import subprocess, sys

def stream_subprocess(cmd):
    """Run cmd, streaming the child's stdout into the notebook live.
    Uses python -u (unbuffered) + line-by-line Popen so Colab shows
    per-step training output instead of swallowing it."""
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    return proc.returncode

cmd = [
    sys.executable, '-u', 'train.py',
    '--config',          'configs/default.yaml',
    '--ckpt_dir',        str(CKPT_DIR),
    '--log_dir',         str(LOG_DIR),
    '--train_manifest',  str(COMBINED_TRAIN_CSV),
    '--val_manifest',    str(COMBINED_VAL_CSV),
    '--resume_from',     'auto',
]
print('Starting Israeli v1 training (clean, no warm-start, auto-resume)...')
print('Command:', ' '.join(cmd))
rc = stream_subprocess(cmd)
print(f'\nExit code: {rc}')

## Cell 5 — Loss curves (train + val overlaid)

**What this does.** Reads TensorBoard scalars and plots train vs val loss plus
the LR schedule; reports best and final validation loss.

**Inputs.** `LOG_DIR` event files.

**Outputs.** `loss_curves.png` under `CKPT_DIR`; best/final val printed.

**Action required.** None.

**Runtime.** seconds.


In [ ]:
# ============================================================
# Cell 5 — Loss curves (train + val overlaid)
# ============================================================

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pathlib

assert 'LOG_DIR' in dir(), 'LOG_DIR not set — re-run Cell 1 first.'

try:
    from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'tensorboard'])
    from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

ea = EventAccumulator(LOG_DIR)
ea.Reload()
available = ea.Tags().get('scalars', [])

def get_scalar(tag):
    if tag not in available: return [], []
    events = ea.Scalars(tag)
    return [e.step for e in events], [e.value for e in events]

train_steps, train_loss = get_scalar('train/loss')
val_steps,   val_loss   = get_scalar('val/loss')
lr_steps,    lr_vals    = get_scalar('train/lr')

best_val, best_step = None, None
if val_loss:
    best_idx  = val_loss.index(min(val_loss))
    best_val  = val_loss[best_idx]
    best_step = val_steps[best_idx]
    print(f'Best val: {best_val:.4f} at step {best_step}')
    print(f'Final val: {val_loss[-1]:.4f} at step {val_steps[-1]}')

fig, (ax_loss, ax_lr) = plt.subplots(1, 2, figsize=(14, 4))

if train_loss:
    ax_loss.plot(train_steps, train_loss, lw=0.8, alpha=0.55, color='steelblue', label='train/loss')
if val_loss:
    ax_loss.plot(val_steps, val_loss, marker='o', ms=3, color='tomato', label='val/loss')
    if best_val is not None:
        ax_loss.axhline(best_val, color='green', ls='--', lw=0.8,
                        label=f'best val={best_val:.4f} @ step {best_step}')
ax_loss.set_xlabel('step'); ax_loss.set_ylabel('loss')
ax_loss.set_title('Train vs. Validation loss')
ax_loss.legend(fontsize=8)

if lr_vals:
    ax_lr.plot(lr_steps, lr_vals, lw=0.8, color='orange')
ax_lr.set_xlabel('step'); ax_lr.set_ylabel('lr')
ax_lr.set_title('Learning rate schedule')

plt.tight_layout()
fig_path = pathlib.Path(CKPT_DIR) / 'loss_curves.png'
plt.savefig(str(fig_path), dpi=120)
plt.show()
print(f'Saved: {fig_path}')

## Cell 5.5 — Mel progression gallery

**What this does.** Assembles the periodic mel snapshot PNGs saved during
training into a single vertical gallery so you can eyeball quality over steps.

**Inputs.** `CKPT_DIR/mel_viz/step_*.png`.

**Outputs.** `mel_gallery.png` under `CKPT_DIR`.

**Action required.** None.

**Runtime.** seconds.


In [ ]:
# ============================================================
# Cell 5.5 — Mel progression gallery
# ============================================================

import pathlib
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

viz_dir = pathlib.Path(CKPT_DIR) / 'mel_viz'

if not viz_dir.exists():
    print(f'No mel_viz directory found at {viz_dir}')
else:
    pngs = sorted(viz_dir.glob('step_*.png'))
    n = len(pngs)
    if n == 0:
        print('No mel snapshot PNGs found.')
    else:
        print(f'{n} mel snapshots found — displaying gallery...')
        fig, axes = plt.subplots(n, 1, figsize=(15, 3.2 * n))
        if n == 1: axes = [axes]
        for ax, png in zip(axes, pngs):
            img = mpimg.imread(str(png))
            ax.imshow(img); ax.axis('off')
            step_num = int(png.stem.replace('step_', ''))
            ax.set_title(f'Step {step_num:,}', fontsize=9)
        plt.suptitle('Mel Progression Gallery', fontsize=12, y=1.0)
        plt.tight_layout()
        gallery_path = pathlib.Path(CKPT_DIR) / 'mel_gallery.png'
        plt.savefig(str(gallery_path), dpi=100, bbox_inches='tight')
        plt.show()
        print(f'Gallery saved: {gallery_path}')

## Cell 6 — Gate check part 1: val loss

**What this does.** Reads `val/loss` from the logs and checks the best value
falls in the expected band [0.15, 0.30]; reports the latest validation sample mel.

**Inputs.** `LOG_DIR`; `CKPT_DIR/samples`.

**Outputs.** Console (pass/fail + best/last val) and latest sample mel info.

**Action required.** None — if in band, proceed to the style-transfer demo.

**Runtime.** seconds.


In [ ]:
# ============================================================
# Cell 6 — Gate check part 1: val loss
# ============================================================

import pathlib, torch
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

ea = EventAccumulator(LOG_DIR); ea.Reload()
val_events = ea.Scalars('val/loss') if 'val/loss' in ea.Tags().get('scalars', []) else []

if val_events:
    last_val = val_events[-1]
    best_val = min(e.value for e in val_events)
    print(f'Last val/loss : {last_val.value:.4f} at step {last_val.step}')
    print(f'Best val/loss : {best_val:.4f}')
    GATE_LO, GATE_HI = 0.15, 0.30
    if GATE_LO <= best_val <= GATE_HI:
        print(f'GATE PART 1 PASSED: best_val {best_val:.4f} ∈ [{GATE_LO}, {GATE_HI}]')
    else:
        print(f'GATE PART 1 NOT MET: best_val {best_val:.4f}')
else:
    print('No val/loss events found.')

sample_dir = pathlib.Path(CKPT_DIR) / 'samples'
mel_files = sorted(sample_dir.glob('step_*_mels.pt')) if sample_dir.exists() else []
if mel_files:
    latest = mel_files[-1]
    mels = torch.load(latest, weights_only=True)
    print(f'\nLatest sample mel: {latest.name}  shape={tuple(mels.shape)}')
    print('→ Run Cell 7 for style transfer demo + F1 evaluation.')

## Cell 7 — Gate check part 2: style-transfer demo

**What this does.** Loads the trained EMA model + BigVGAN vocoder, takes one
Israeli val piano roll and renders one audio clip per style (version 0 vs 1 vs 2)
from the SAME conditioning — they must sound noticeably different.

**Inputs.** `CKPT_DIR/best_val.pt`, `configs/default.yaml`, `P_ARTISTS.val_csv`
+ score tensors, BigVGAN weights.

**Outputs.** `version_<id>_<label>.wav` under `CKPT_DIR/style_transfer_demo`;
inline audio players + a manual checklist.

**Action required.** ⚠ Listen — confirm each output is musical and the styles
are distinguishable from one another.

**Runtime.** ~1–2 min.

In [ ]:
# ============================================================
# Cell 7 — Style transfer demo
# Takes one val segment from the Israeli (Artists) split and generates
# one audio output per style defined in STYLES (Cell 1).
#
# Adding a new style: update STYLES + STYLE_DATA in Cell 1,
# re-train, then re-run this cell — no changes needed here.
#
# Gate: each pair of outputs must sound noticeably different.
# ============================================================

import csv, pathlib, sys
import numpy as np
import torch
import soundfile as sf
from IPython.display import Audio, display
from omegaconf import OmegaConf

sys.path.insert(0, '/content/MusicProject')
from model.unet import UNet1D
from model.diffusion import GaussianDiffusion
from postprocessing.bigvgan_vocoder import BigVGANVocoder

cfg    = OmegaConf.load('configs/default.yaml')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

assert cfg.conditioning.n_versions == N_VERSIONS, (
    f'Config n_versions={cfg.conditioning.n_versions} does not match '
    f'N_VERSIONS={N_VERSIONS} from Cell 1. Update configs/default.yaml and re-run Cell 1.'
)

ckpt_path = pathlib.Path(CKPT_DIR) / 'best_val.pt'
assert ckpt_path.exists(), f'best_val.pt not found at {ckpt_path}'
state = torch.load(ckpt_path, map_location=device, weights_only=False)

model = UNet1D(
    mel_channels       = cfg.model.mel_channels,
    score_channels     = cfg.model.score_channels,
    base_channels      = cfg.model.base_channels,
    channel_mults      = list(cfg.model.channel_mults),
    num_res_blocks_enc = cfg.model.num_res_blocks_enc,
    num_res_blocks_dec = cfg.model.num_res_blocks_dec,
    attention_levels   = list(cfg.model.attention_levels),
    attn_heads         = cfg.model.attention_heads,
    n_groups           = cfg.model.n_groups,
    dropout            = cfg.model.dropout,
    n_versions         = cfg.conditioning.n_versions,
    version_emb_dim    = cfg.conditioning.version_emb_dim,
    time_emb_dim       = cfg.conditioning.time_emb_dim,
).to(device).eval()

weights_key = 'ema' if 'ema' in state else 'model'
model.load_state_dict(state[weights_key], strict=True)
print(f'Loaded {weights_key} weights from {ckpt_path.name}')
print(f'Styles to generate: {STYLES}')

diffusion = GaussianDiffusion(
    model       = model,
    T           = cfg.diffusion.T_train,
    n_versions  = cfg.conditioning.n_versions,
    cfg_score   = cfg.sampling.cfg_score,
    cfg_version = cfg.sampling.cfg_version,
).to(device)

# Load one Israeli val segment as the source piano roll.
# Israeli split CSV paths include the 'processed_data/' prefix and are relative
# to version_dir (NOT processed_data_dir — that would double the prefix).
with open(P_ARTISTS.val_csv, newline='') as f:
    row = list(csv.DictReader(f))[0]

score_pt = P_ARTISTS.version_dir / row['score_path']
score    = torch.load(score_pt, weights_only=True).unsqueeze(0).to(device)
score    = score.view(1, -1, score.shape[-1])   # [1, 256, 430]
print(f'\nSource: song_id={row["song_id"]}  segment={row["segment_idx"]}')
print('Same piano roll → all styles below\n')

voc = BigVGANVocoder(model_name='bigvgan_22k', device=str(device))
eval_dir = pathlib.Path(CKPT_DIR) / 'style_transfer_demo'
eval_dir.mkdir(exist_ok=True)

# Generate one output per style — loop auto-extends when new styles are added
for ver_id, label in STYLES.items():
    vid = torch.tensor([ver_id], dtype=torch.long, device=device)
    with torch.no_grad():
        gen_mel = diffusion.ddim_sample(score, vid,
                                        N=cfg.sampling.N_ddim,
                                        overlap_frames=cfg.sampling.overlap_frames)
    gen_mel_cpu = gen_mel.squeeze(0).cpu().numpy()
    mel_db      = (gen_mel_cpu + 1.0) / 2.0 * 80.0 - 80.0
    mel_logmag  = (mel_db * (np.log(10) / 20.0)).astype(np.float32)
    audio       = voc.mel_to_audio(torch.from_numpy(mel_logmag).unsqueeze(0))
    wav_path    = eval_dir / f'version_{ver_id}_{label}.wav'
    sf.write(str(wav_path), audio, voc.h.sampling_rate)
    print(f'--- version_id={ver_id}  ({label}) ---')
    display(Audio(str(wav_path)))

print()
print('STYLE TRANSFER CHECKLIST:')
for ver_id, label in STYLES.items():
    print(f'  [ ] version_id={ver_id} ({label}) sounds musical and matches expected style')
print('  [ ] All outputs are noticeably different from each other in timbre/style')

## Cell 7b — Round-trip confirmation: is the mel→BigVGAN conversion the audio bottleneck?

**What this does.** Removes the diffusion model from the loop. It vocodes the
**ground-truth stored training mel** (the best output the model could ever match)
through the *exact* conversion Cell 7 uses. If this sounds as bad as the style
demos, the quality loss is in the mel→BigVGAN conversion / domain — **not** the
model or the conditioning.

**Inputs.** `P_ARTISTS.val_csv`, `voc` and `CKPT_DIR` (from Cell 7).

**Outputs.** `roundtrip_A_currentconv.wav` under `CKPT_DIR/style_transfer_demo`,
plus a printed comparison of the log-magnitude range fed to BigVGAN vs. its
expected native range.

**Action required.** Run Cell 7 first, then run this cell and listen. If Path A
already sounds degraded, the model is fine — the fix is the preprocessing/vocoder
mel convention (BigVGAN-native mel + fixed-constant normalization), which needs
re-preprocessing + retrain for clean audio.

**Runtime.** ~20–30 s (a single vocode, no diffusion sampling).

In [ ]:
# ============================================================
# Cell 7b — ROUND-TRIP CONFIRMATION (no retrain)
# Vocode the GROUND-TRUTH target mel through the EXACT Cell-7 conversion.
# Decisive test: model removed from the loop.
# ============================================================
import csv, pathlib
import numpy as np
import torch
import soundfile as sf
from IPython.display import Audio, display

demo_dir = pathlib.Path(CKPT_DIR) / 'style_transfer_demo'
demo_dir.mkdir(exist_ok=True)

# Reuse the same val segment Cell 7 used as the source.
with open(P_ARTISTS.val_csv, newline='') as f:
    row = list(csv.DictReader(f))[0]

# Stored TARGET mel: [80, 430], normalized to [-1, 1] — same domain the model emits.
mel_pt   = P_ARTISTS.version_dir / row['segment_path']
mel_norm = torch.load(mel_pt, weights_only=True).float().numpy()
print(f'Ground-truth target mel : {mel_pt.name}')
print(f'  shape={mel_norm.shape}  norm-range=[{mel_norm.min():.3f}, {mel_norm.max():.3f}]')

# ---- Path A: EXACT current Cell-7 conversion (the suspect) ------------------
mel_db_A     = (mel_norm + 1.0) / 2.0 * 80.0 - 80.0           # [-1,1] -> [-80,0] dB
mel_logmag_A = (mel_db_A * (np.log(10) / 20.0)).astype(np.float32)
audio_A      = voc.mel_to_audio(torch.from_numpy(mel_logmag_A).unsqueeze(0))
sf.write(str(demo_dir / 'roundtrip_A_currentconv.wav'), audio_A, voc.h.sampling_rate)

# ---- Domain check: what BigVGAN is actually being fed -----------------------
print('\n[domain check] log-magnitude actually fed to BigVGAN:')
print(f'  current conversion -> [{mel_logmag_A.min():.2f}, {mel_logmag_A.max():.2f}]')
print('  BigVGAN native mels normally span roughly [-12, +2]')
print('  (log of magnitude; clamp floor ln(1e-5) = -11.51)')

print('\n--- Path A: GROUND-TRUTH mel through the CURRENT conversion ---')
print('    This is the CEILING — the best the model could ever sound.')
print('    If it is bad, the model is NOT the problem; the conversion is.')
display(Audio(str(demo_dir / 'roundtrip_A_currentconv.wav')))


## Cell 7c — Checkpoint sweep: hear all 3 styles across SEVERAL training checkpoints

**Why.** To judge whether training actually succeeded, listen to the *same*
source segment generated by *different* checkpoints (e.g. `best_val.pt` plus a
spread of `step_*.pt`). If later/best checkpoints sound clearly better and the
3 styles stay distinct, training converged; if quality is flat or noisy across
all of them, the limit is the model/data, not the demo.

**What this does.** For each checkpoint in `CKPT_STEPS`, loads its **EMA** weights
into `model`, runs `ddim_sample` for all 3 styles on `N_TRACKS` fixed source
segment(s), vocodes via the validated Direct-BigVGAN path (clean per Cell 7b),
and prints each checkpoint's step. Players are grouped per checkpoint.

**Inputs.** `model`, `diffusion`, `voc`, `STYLES`, `P_ARTISTS`, `cfg`, `CKPT_DIR`,
`device` from Cell 7 (run Cell 7 first; needs a GPU — it samples once per
checkpoint × style × track). Tunables at the top of the cell: `CKPT_STEPS`
(which checkpoints), `N_TRACKS`, `CFG_SCORE`, `CFG_VERSION`, `N_DDIM`.

**Outputs.** WAVs under `CKPT_DIR/checkpoint_sweep_demo`, named
`ckpt{step}__track{i}_{song}_seg{seg}__version_{id}_{label}.wav`.

**Action required.** Run Cell 7 first, set `CKPT_STEPS` to the checkpoints you
want to compare, then run this cell and listen per checkpoint.

**Runtime.** GPU; roughly `len(CKPT_STEPS) × 3 styles × N_TRACKS` samples — a few
minutes for a small sweep, longer as you add checkpoints.

In [ ]:
# ============================================================
# Cell 7c — CHECKPOINT SWEEP (175k -> 250k)
# Same source + same 3 styles, generated by each checkpoint's EMA weights.
# Vocoding = the validated Direct-BigVGAN path (confirmed clean in Cell 7b).
# Reuses model/diffusion/voc/STYLES/P_ARTISTS/cfg/device from Cell 7.
# ============================================================
import csv, pathlib
import numpy as np
import torch
import soundfile as sf
from IPython.display import Audio, display

# ---- tunables ---------------------------------------------------------------
STEP_LO, STEP_HI = 175_000, 250_000   # checkpoint step range (inclusive)
STEP_STRIDE      = 1                   # 1 = every checkpoint found; raise to thin
N_TRACKS         = 1                   # distinct source segments from the val split
CFG_SCORE        = cfg.sampling.cfg_score      # default 1.25 (override to sweep)
CFG_VERSION      = cfg.sampling.cfg_version    # default 1.25
N_DDIM           = cfg.sampling.N_ddim         # default 100
OVERLAP          = cfg.sampling.overlap_frames

sweep_dir = pathlib.Path(CKPT_DIR) / 'checkpoint_sweep_demo'
sweep_dir.mkdir(exist_ok=True)

# ---- discover checkpoints in range ------------------------------------------
def _step_of(p):
    try:
        return int(p.stem.split('_')[1])
    except (IndexError, ValueError):
        return None

all_ckpts = []
for p in pathlib.Path(CKPT_DIR).glob('step_*.pt'):
    s = _step_of(p)
    if s is not None and STEP_LO <= s <= STEP_HI:
        all_ckpts.append((s, p))
all_ckpts.sort(key=lambda t: t[0])
all_ckpts = all_ckpts[::STEP_STRIDE]

# Always include best_val.pt (labelled by its own stored step) for reference.
best_path = pathlib.Path(CKPT_DIR) / 'best_val.pt'

print(f'Checkpoints in [{STEP_LO:,}, {STEP_HI:,}] (stride {STEP_STRIDE}): '
      f'{[s for s, _ in all_ckpts]}')
print(f'Total sampling runs: {len(all_ckpts)} ckpts x {N_TRACKS} tracks '
      f'x {len(STYLES)} styles = {len(all_ckpts) * N_TRACKS * len(STYLES)}')
print(f'Sampling: N_ddim={N_DDIM}, cfg_score={CFG_SCORE}, cfg_version={CFG_VERSION}\n')

if not all_ckpts:
    raise FileNotFoundError(
        f'No step_*.pt found in [{STEP_LO}, {STEP_HI}] under {CKPT_DIR}. '
        f'Check the range or that keep-all checkpoints are present.')

# ---- load the fixed source segment(s) once ----------------------------------
with open(P_ARTISTS.val_csv, newline='') as f:
    rows = list(csv.DictReader(f))[:N_TRACKS]

sources = []
for row in rows:
    score_pt = P_ARTISTS.version_dir / row['score_path']
    sc = torch.load(score_pt, weights_only=True).unsqueeze(0).to(device)
    sc = sc.view(1, -1, sc.shape[-1])              # [1, 256, 430]
    sources.append((row.get('song_id', '?'), row.get('segment_idx', '?'), sc))
print(f'Source segments: {[(s, seg) for s, seg, _ in sources]}\n')

# ---- helper: model EMA mel -> audio via the validated path ------------------
def _mel_to_audio(gen_mel):
    mel = gen_mel.squeeze(0).cpu().numpy()
    mel_db     = (mel + 1.0) / 2.0 * 80.0 - 80.0
    mel_logmag = (mel_db * (np.log(10) / 20.0)).astype(np.float32)
    return voc.mel_to_audio(torch.from_numpy(mel_logmag).unsqueeze(0))

# ---- sweep ------------------------------------------------------------------
for step, ckpt_path in all_ckpts:
    st = torch.load(ckpt_path, map_location=device, weights_only=False)
    wkey = 'ema' if 'ema' in st else 'model'
    model.load_state_dict(st[wkey], strict=True)
    model.eval()
    print(f'================  checkpoint step {step:,}  ({wkey} weights)  ================')

    for t_idx, (song_id, seg_idx, score) in enumerate(sources):
        print(f'--- track {t_idx}: song_id={song_id} seg={seg_idx} ---')
        for ver_id, label in STYLES.items():
            vid = torch.tensor([ver_id], dtype=torch.long, device=device)
            with torch.no_grad():
                gen_mel = diffusion.ddim_sample(
                    score, vid, N=N_DDIM,
                    cfg_score=CFG_SCORE, cfg_version=CFG_VERSION,
                    overlap_frames=OVERLAP)
            audio = _mel_to_audio(gen_mel)
            wav_path = (sweep_dir /
                        f'ckpt{step}__track{t_idx}_{song_id}_seg{seg_idx}'
                        f'__version_{ver_id}_{label}.wav')
            sf.write(str(wav_path), audio, voc.h.sampling_rate)
            print(f'    version_id={ver_id} ({label})')
            display(Audio(str(wav_path)))

print('\nDone. Compare across checkpoints: later/best should sound cleaner while '
      'the 3 styles stay distinct -> training converged.')


## Cell 7d — Sampling grid: CFG × DDIM on the Cell-7c winner checkpoint(s)

**Why.** Once Cell 7c reveals the best-sounding checkpoint(s) per ear, this cell
sweeps the two no-retrain sampling knobs — **CFG strength** and **DDIM steps** —
on those exact checkpoints to see if either sharpens the audio. CFG is usually
the bigger lever; steps (100→200) is a cheap secondary check.

**What this does.** For each `step` in `WINNER_STEPS`, loads its EMA weights and
renders all 3 styles for every `(cfg, n_ddim)` combo in the grid, vocoding via
the validated Direct-BigVGAN path. Players are grouped per checkpoint → per
setting so you can A/B directly.

**Inputs.** `model`, `diffusion`, `voc`, `STYLES`, `P_ARTISTS`, `cfg`, `CKPT_DIR`,
`device` from Cell 7 (run Cell 7 first; needs a GPU). Tunables at the top of the
cell: `WINNER_STEPS`, `CFG_GRID`, `NDDIM_GRID`, `N_TRACKS`. Default grid =
current (1.25/100), stronger CFG (2.5/100), more steps (1.25/200), both (2.5/200).

**Outputs.** WAVs under `CKPT_DIR/sampling_grid_demo`, named
`ckpt{step}__cfg{cfg}_ddim{N}__version_{id}_{label}.wav`.

**Action required.** Run Cell 7 first and set `WINNER_STEPS` from your Cell-7c
listening, then run this cell and A/B the grid.

**Runtime.** GPU; `len(WINNER_STEPS) × len(CFG_GRID) × len(NDDIM_GRID) × 3 styles`
samples — trim the grids to keep it manageable.

In [ ]:
# ============================================================
# Cell 7d — DEFENCE DEMO: all marked winner checkpoints
# For each checkpoint x sampling-setting x style:
#   - generate mel, vocode (validated Direct-BigVGAN path)
#   - VISUALIZE the mel (3 styles side-by-side)
#   - play the audio
#   - save .wav + .png and append to a manifest.csv
# Reuses model/diffusion/voc/STYLES/P_ARTISTS/cfg/device from Cell 7.
# ============================================================
import csv, pathlib
import numpy as np
import torch
import soundfile as sf
import matplotlib.pyplot as plt
from IPython.display import Audio, display, Markdown

# ---- the checkpoints you marked as winners ----------------------------------
# (value in steps; comment = your listening note)
WINNER_STEPS = [
    190_000,   # good
    202_000,   # good
    204_000,   # only Israeli artists
    212_000,   # best so far
    220_000,   # also good, pretty smooth
    224_000,   # distinct differences + similarities to originals
    232_000,   # best mel similarity (hearing not so close)
    234_000,   # best military bands so far
    238_000,   # clear difference in singers (keep)
    242_000,   # keep
    248_000,   # keep
]

# ---- sampling grid: (cfg_score, cfg_version) pairs  x  DDIM step counts ------
# Trim these to shrink runtime. Full 2x2 grid below = current / stronger / more / both.
CFG_GRID   = [(1.25, 1.25), (2.5, 2.5)]    # current vs stronger
NDDIM_GRID = [100, 200]                      # current vs more steps
N_TRACKS   = 1                               # source segments from the val set
OVERLAP    = cfg.sampling.overlap_frames

grid_dir = pathlib.Path(CKPT_DIR) / 'sampling_grid_demo'
grid_dir.mkdir(exist_ok=True)
manifest_path = grid_dir / 'manifest.csv'

# ---- resolve each requested step to the NEAREST existing step_*.pt -----------
_avail = []
for p in pathlib.Path(CKPT_DIR).glob('step_*.pt'):
    try:
        _avail.append((int(p.stem.split('_')[1]), p))
    except (IndexError, ValueError):
        pass
_avail.sort()
if not _avail:
    raise FileNotFoundError(f'No step_*.pt checkpoints found under {CKPT_DIR}')

def _resolve_ckpt(target):
    step_num, path = min(_avail, key=lambda sp: abs(sp[0] - target))
    if step_num != target:
        print(f'  requested {target:,} -> nearest available {step_num:,} ({path.name})')
    return step_num, path

resolved = [_resolve_ckpt(s) for s in WINNER_STEPS]

n_runs = len(resolved) * len(CFG_GRID) * len(NDDIM_GRID) * len(STYLES) * N_TRACKS
print(f'Winners: {[s for s, _ in resolved]}')
print(f'CFG grid: {CFG_GRID}   DDIM grid: {NDDIM_GRID}')
print(f'Total DDIM sampling runs: {n_runs}  (200-step runs are ~2x slower)\n')

# ---- load fixed source segment(s) once (same source for every render) --------
with open(P_ARTISTS.val_csv, newline='') as f:
    rows = list(csv.DictReader(f))[:N_TRACKS]
sources = []
for row in rows:
    score_pt = P_ARTISTS.version_dir / row['score_path']
    sc = torch.load(score_pt, weights_only=True).unsqueeze(0).to(device)
    sc = sc.view(1, -1, sc.shape[-1])
    sources.append((row.get('song_id', '?'), row.get('segment_idx', '?'), sc))

def _mel_to_audio(gen_mel):
    """Validated Direct-BigVGAN conversion (clean per Cell 7b)."""
    mel = gen_mel.squeeze(0).cpu().numpy()
    mel_db     = (mel + 1.0) / 2.0 * 80.0 - 80.0
    mel_logmag = (mel_db * (np.log(10) / 20.0)).astype(np.float32)
    audio = voc.mel_to_audio(torch.from_numpy(mel_logmag).unsqueeze(0))
    return audio, mel

# ---- manifest + sweep --------------------------------------------------------
manifest_rows = []

for step, ckpt_path in resolved:
    st = torch.load(ckpt_path, map_location=device, weights_only=False)
    wkey = 'ema' if 'ema' in st else 'model'
    model.load_state_dict(st[wkey], strict=True)
    model.eval()
    display(Markdown(f'# checkpoint step {step:,}  ({wkey})'))

    for (w_s, w_v) in CFG_GRID:
        for n_ddim in NDDIM_GRID:
            display(Markdown(
                f'**step {step:,} | cfg_score={w_s} | cfg_version={w_v} | DDIM={n_ddim}**'))
            for t_idx, (song_id, seg_idx, score) in enumerate(sources):
                mels, audios = [], []
                for ver_id, label in STYLES.items():
                    vid = torch.tensor([ver_id], dtype=torch.long, device=device)
                    with torch.no_grad():
                        gen_mel = diffusion.ddim_sample(
                            score, vid, N=n_ddim,
                            cfg_score=w_s, cfg_version=w_v,
                            overlap_frames=OVERLAP)
                    audio, mel = _mel_to_audio(gen_mel)
                    tag = (f'ckpt{step}__cfg{w_s}_ddim{n_ddim}'
                           f'__track{t_idx}_{song_id}_seg{seg_idx}'
                           f'__version_{ver_id}_{label}')
                    wav_path = grid_dir / f'{tag}.wav'
                    sf.write(str(wav_path), audio, voc.h.sampling_rate)
                    mels.append((ver_id, label, mel))
                    audios.append((label, wav_path))
                    manifest_rows.append({
                        'step': step, 'cfg_score': w_s, 'cfg_version': w_v,
                        'ddim': n_ddim, 'version_id': ver_id, 'label': label,
                        'track': t_idx, 'song_id': song_id, 'segment_idx': seg_idx,
                        'wav': str(wav_path),
                    })

                # --- visualize the 3 style mels side-by-side ------------------
                fig, axes = plt.subplots(1, len(mels), figsize=(4.6 * len(mels), 3.2))
                if len(mels) == 1:
                    axes = [axes]
                im = None
                for ax, (ver_id, label, mel) in zip(axes, mels):
                    im = ax.imshow(mel, origin='lower', aspect='auto',
                                   interpolation='nearest', vmin=-1, vmax=1, cmap='magma')
                    ax.set_title(f'v{ver_id} - {label}', fontsize=10)
                    ax.set_xlabel('frames'); ax.set_ylabel('mel bins')
                fig.suptitle(f'step {step:,} | cfg {w_s} | DDIM {n_ddim}', fontsize=11)
                fig.colorbar(im, ax=axes, fraction=0.025, pad=0.01)
                png_path = grid_dir / (f'ckpt{step}__cfg{w_s}_ddim{n_ddim}'
                                       f'__track{t_idx}_mels.png')
                fig.savefig(str(png_path), dpi=90, bbox_inches='tight')
                plt.show()

                # --- audio players, one per style -----------------------------
                for label, wav_path in audios:
                    display(Markdown(f'&nbsp;&nbsp;**{label}**'))
                    display(Audio(str(wav_path)))

# ---- write manifest ----------------------------------------------------------
with open(manifest_path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(manifest_rows[0].keys()))
    w.writeheader()
    w.writerows(manifest_rows)

print(f'\nDone. {len(manifest_rows)} clips written to: {grid_dir}')
print(f'Manifest (step/cfg/ddim/version -> wav): {manifest_path}')
print('PNG mel grids saved alongside each setting for slides.')
